In [0]:
# Install required libraries
# azure-storage-blob: connects to Azure Blob Storage
# openpyxl: required by pandas to read .xlsx Excel files
%pip install azure-storage-blob openpyxl

In [0]:
# Restart kernel after install to load new libraries
dbutils.library.restartPython()

##  Connect & load synergysuite raw file

In [0]:
# ============================================
# Connect to Azure Blob Storage
# Load SynergySuite raw Excel from bronze
# ============================================
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io

# Connect securely via Key Vault — never hardcode credentials
storage_account_key = dbutils.secrets.get(scope="churchs-payroll-kv", key="storage-account-key")
blob_service_client = BlobServiceClient(
    account_url="https://churchspayrollstorage.blob.core.windows.net",
    credential=storage_account_key
)
print("Connected to Azure Blob Storage successfully!")

# Point to SynergySuite file in bronze container
# Note: folder name has trailing space — 'synergysuite '
blob_client = blob_service_client.get_blob_client(
    container="bronze",
    blob="synergysuite /SynergySuite Input.xlsx"
)

# Download and read Excel without headers
# header=None because file has complex multi-row header structure
blob_data = blob_client.download_blob().readall()
df_raw = pd.read_excel(io.BytesIO(blob_data), header=None)

# Validation — check file loaded correctly
print("Raw file loaded! Shape:", df_raw.shape)
assert df_raw.shape[0] > 0, "ERROR: Raw file is empty!"
assert df_raw.shape[1] > 0, "ERROR: No columns found!"
print("Validation passed — file has", df_raw.shape[0], "rows and", df_raw.shape[1], "columns")

## Parse hierarchy 

In [0]:
# ============================================
# Parse SynergySuite hierarchy
# Structure: Store → Employee → Position → Shift rows
# No standard headers — must detect row types manually
# ============================================

records = []
store = employee = payroll = position = None

# Track counts for logging
store_count = 0
employee_count = 0
skipped_rows = 0

for idx, row in df_raw.iterrows():
    b = str(row[1]).strip() if pd.notna(row[1]) else ""

    # Store row — identified by FC and dash in name
    if "FC" in b and "-" in b:
        store = b
        store_count += 1

    # Employee row — identified by Employee / Payroll label
    elif "Employee / Payroll" in b:
        info = str(row[5]).strip() if pd.notna(row[5]) else ""
        if "/" in info:
            employee, payroll = [x.strip() for x in info.split("/")]
            employee_count += 1

    # Position row
    elif b == "Position":
        position = str(row[2]).strip() if pd.notna(row[2]) else ""

    # Shift data row — identified by numeric rate in column 3
    elif pd.notna(row[3]) and str(row[3]).replace('.','').isdigit():
        try:
            records.append({
                "StoreID":       store,
                "Employee_name": employee,
                "Payroll_ID":    payroll,
                "Position":      position,
                "Shift_Date":    pd.to_datetime(b, errors="coerce"),
                "Rate":          row[3],
                "Regular_Hours": pd.to_numeric(row[6], errors="coerce"),
                "Regular_Labor": row[9],
                "OT_Hours":      row[10],
                "OT_Labor":      row[11],
                "Total_Hours":   row[14],
                "Total_Labor":   row[15],
                "Non_Cash_Tips": row[18],
                "Declared_Tips": row[19]
            })
        except Exception as e:
            skipped_rows += 1
            print(f"Warning: Skipped row {idx} — {e}")

# Build silver DataFrame
df_silver = pd.DataFrame(records)

# Add week ending date — Saturday of each shift week
df_silver["week_end_date"] = df_silver["Shift_Date"].apply(
    lambda d: d + pd.Timedelta(days=(5 - d.weekday()) % 7) if pd.notna(d) else None
)

# Logs
print("\n=== Parse Summary ===")
print("Stores found:", store_count)
print("Employees found:", employee_count)
print("Shift records parsed:", len(records))
print("Rows skipped:", skipped_rows)

# Validation checks
print("\n=== Validation Checks ===")
assert len(df_silver) > 0, "ERROR: No records parsed!"
assert df_silver["StoreID"].notna().all(), "ERROR: Missing StoreID values!"
assert df_silver["Employee_name"].notna().all(), "ERROR: Missing Employee names!"
assert df_silver["Payroll_ID"].notna().all(), "ERROR: Missing Payroll IDs!"
assert df_silver["Shift_Date"].notna().sum() > 0, "ERROR: No valid shift dates!"
print("All validation checks passed!")

print("\nUnique stores:", df_silver["StoreID"].nunique())
print("Unique employees:", df_silver["Payroll_ID"].nunique())
print("Date range:", df_silver["Shift_Date"].min(), "to", df_silver["Shift_Date"].max())
print("\nSample:")
print(df_silver.head(5).to_string())

## SynergySuite Bronze → Silver

In [0]:
# ============================================
# Upload cleaned SynergySuite data
# to silver container as CSV
# ============================================

# Convert DataFrame to CSV in memory
csv_buffer = io.BytesIO()
df_silver.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

# Upload to silver container
silver_client = blob_service_client.get_blob_client(
    container="silver",
    blob="synergysuite/SynergySuite_Silver.csv"
)
silver_client.upload_blob(csv_buffer, overwrite=True)

# Validation — verify upload
print("SynergySuite Silver CSV uploaded successfully!")
print("Location: silver/synergysuite/SynergySuite_Silver.csv")
print("Total records written:", len(df_silver))
print("Columns written:", df_silver.columns.tolist())
assert len(df_silver) > 0, "ERROR: Nothing was written to silver!"
print("Upload validation passed!")